# **Durability — NN validation plots**

Reads the NN models/scaler written by [`04_train_nn.ipynb`](04_train_nn.ipynb) and the stage 1
datasets written by [`01_generate_dataset.ipynb`](01_generate_dataset.ipynb) — makes **no** training
calls, only loads and plots. Three things:

1. predicted vs. true lambda 1 / lambda 2, on the held-out validation split;
2. a spot check — load the model, fix `(fck, rh, cov)`, sweep $t$, predict; and
3. a KL-divergence check of the NN-predicted GLD against the emulator's own direct fit, at one
   design point — same convention as [`02_train_pce_plot.ipynb`](02_train_pce_plot.ipynb).

Same figure conventions throughout: no titles (captions belong in the LaTeX), configurable
size/fonts/dpi, each saved next to the data it came from.

## 1. Libraries

In [1]:
%matplotlib inline
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import dill
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
mpl.rcParams.update({
                        'font.family': 'serif',
                        'mathtext.fontset': 'cm',
                        'axes.unicode_minus': False
                    })
from sklearn.model_selection import train_test_split

from functions import *

C:\git-projetos\2024-1_victor_hugo_renata_maria\.venv\Lib\site-packages\UQpy\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


## 2. Config

`n_latent_samples`, `installation_year`, `co2_scenario`, `cement_type`, `exposure_conditions`,
`feature_cols`, `target_cols`, `test_frac` and `random_state` must match
[`04_train_nn.ipynb`](04_train_nn.ipynb) — together they name the files being loaded and rebuild the
same held-out validation split.

In [2]:
n_latent_samples    = 2500      # must match stage 3/4 — it names the files being loaded
installation_year   = 1980
co2_scenario        = "SSP2-4.5"
cement_type         = 3
exposure_conditions = 2

feature_cols = ['fck', 'rh', 'cov', 'Time (years)']
target_cols  = ['lambda 1', 'lambda 2']

test_frac    = 0.2   # must match stage 4 — rebuilds the same validation split
random_state = 42    # must match stage 4

fig_size   = (5, 4)      # size of each individual figure, in inches
fig_format = 'png'       # format each figure is saved in ('pdf', 'png', ...)
fig_dpi    = 300         # resolution the figure is saved at (dots per inch)

label_fontsize = 14   # font size of the axis labels
tick_fontsize  = 12   # font size of the tick numbers

xlim = None   # e.g. (-5, 5) to fix the axis; None = auto-scaled to the data, per lambda
ylim = None   # e.g. (-5, 5) to fix the axis; None = auto-scaled to the data, per lambda

## 3. Load the stacked dataset and the trained NN

In [3]:
tag = f'install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'

with open(f'{n_latent_samples}_dataset_nn_durability_{tag}.pkl', 'rb') as f:
    df_nn = dill.load(f)

models = {}
for target in target_cols:
    key = target.replace(' ', '_')
    with open(f'{n_latent_samples}_nn_{key}_model_durability_{tag}.pkl', 'rb') as f:
        models[target] = dill.load(f)

with open(f'{n_latent_samples}_nn_scaler_durability_{tag}.pkl', 'rb') as f:
    scaler = dill.load(f)

print(f"Loaded {len(df_nn)} rows and {len(models)} NN models")
df_nn.head()

Loaded 25000 rows and 2 NN models


         fck         rh        cov  ...  lambda 2  lambda 3  lambda 4
0  40.822752  60.645084  50.982333  ...  1.456656  0.132011  0.141281
1  27.044662  79.374581  52.166163  ...  1.428932  0.132011  0.141281
2  36.577721  24.002504  44.897269  ...  1.632025  0.132011  0.141281
3  22.765737  38.726335  40.107304  ...  1.803812  0.132011  0.141281
4  29.457463  47.761799  19.629058  ...  3.742957  0.132011  0.141281

[5 rows x 8 columns]

## 4. Predicted vs. true, on the held-out validation split

In [4]:
df_clean = df_nn.dropna(subset=target_cols).reset_index(drop=True)
X = df_clean[feature_cols].to_numpy()
y = df_clean[target_cols].to_numpy()
_, x_val, _, y_val = train_test_split(X, y, test_size=test_frac, random_state=random_state)
x_val_s = scaler.transform(x_val)

pkl_name = f'{n_latent_samples}_dataset_nn_durability_{tag}'
figs     = {}

for target in target_cols:
    y_true = y_val[:, target_cols.index(target)]
    y_pred = models[target].predict(x_val_s)
    idx    = target.split(' ')[1]

    fig, ax = plt.subplots(figsize=fig_size)
    ax.scatter(y_true, y_pred, s=8, alpha=0.4, color='0.25')
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_xlabel(f'True $\\lambda_{{{idx}}}$', fontsize=label_fontsize)
    ax.set_ylabel(f'Predicted $\\lambda_{{{idx}}}$', fontsize=label_fontsize)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    fig.savefig(f'{pkl_name}_pred_vs_true_lambda_{idx}.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
    figs[target] = fig
    plt.show()

### 4.1 Mesmo gráfico, em português

Para uso no artigo. Salvo como `<pkl_name>_pred_vs_true_lambda_<idx>_pt.<fig_format>`.

In [5]:
for target in target_cols:
    y_true = y_val[:, target_cols.index(target)]
    y_pred = models[target].predict(x_val_s)
    idx    = target.split(' ')[1]

    fig, ax = plt.subplots(figsize=fig_size)
    ax.scatter(y_true, y_pred, s=8, alpha=0.4, color='0.25')
    lims = [min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())]
    ax.plot(lims, lims, 'r--', lw=1)
    ax.set_xlabel(f'Valor real de $\\lambda_{{{idx}}}$', fontsize=label_fontsize)
    ax.set_ylabel(f'Valor previsto de $\\lambda_{{{idx}}}$', fontsize=label_fontsize)
    ax.tick_params(axis='both', labelsize=tick_fontsize)
    if xlim is not None:
        ax.set_xlim(xlim)
    if ylim is not None:
        ax.set_ylim(ylim)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    fig.savefig(f'{pkl_name}_pred_vs_true_lambda_{idx}_pt.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
    figs[f'{target}_pt'] = fig
    plt.show()

## 5. Spot check — load the model, fix $(f_{ck}, RH, c)$, sweep $t$

Fix a design point and sweep $t$, to see the predicted lambdas evolve smoothly with time.

In [6]:
fck_fixed   = 35.0
rh_fixed    = 50.0
cov_fixed   = 35.0
times_sweep = np.linspace(0, 100, 10, endpoint=True)

sweep_df = pd.DataFrame({
                           'fck':          [fck_fixed] * len(times_sweep),
                           'rh':           [rh_fixed] * len(times_sweep),
                           'cov':          [cov_fixed] * len(times_sweep),
                           'Time (years)': times_sweep,
                        })

x_sweep_s = scaler.transform(sweep_df[feature_cols].to_numpy())
for target in target_cols:
    sweep_df[f'pred_{target}'] = models[target].predict(x_sweep_s)

sweep_df

    fck    rh   cov  Time (years)  pred_lambda 1  pred_lambda 2
0  35.0  50.0  35.0      0.000000      34.620529       2.073296
1  35.0  50.0  35.0     11.111111      29.516619       1.863105
2  35.0  50.0  35.0     22.222222      24.811158       1.797023
3  35.0  50.0  35.0     33.333333      22.216715       1.760494
4  35.0  50.0  35.0     44.444444      19.687544       1.678297
5  35.0  50.0  35.0     55.555556      17.122520       1.631219
6  35.0  50.0  35.0     66.666667      15.407685       1.534011
7  35.0  50.0  35.0     77.777778      14.805805       1.463635
8  35.0  50.0  35.0     88.888889      13.700818       1.422316
9  35.0  50.0  35.0    100.000000      12.171830       1.402599

## 6. NN validation via KL divergence

Spot-checks one design point directly against the NN: the emulator already fits a GLD to its raw
Monte Carlo `g` samples (`dataset_unique`/`dataset_full` from
[`01_generate_dataset.ipynb`](01_generate_dataset.ipynb)) — this asks the NN to predict `lambda 1` /
`lambda 2` from `(fck, rh, cov, t)` alone and scores how close the resulting GLD is to the real one.

`time_index_kl` and `design_point_index_kl` follow the same convention as
[`02_train_pce_plot.ipynb`](02_train_pce_plot.ipynb) — pick the same values here as there to compare
the PCE and the NN at exactly the same design point.

In [7]:
time_index_kl         = 2   # index into `times_kl` — which stage-1 time step to check
design_point_index_kl = 0   # row index into that time step's dataset_unique_train — which design point to check

times_kl = np.linspace(0, 100, 5, endpoint=True)   # must match stage 1

lambda3_kl = 0.132011   # written by hand — the NN never predicts lambda 3 / lambda 4
lambda4_kl = 0.141281

n_grid_kl = 400   # grid points used to numerically integrate the KL divergence and R²

xlim_kl = None   # e.g. (-5, 30) to fix the g axis; None = auto-scaled to the data
ylim_kl = None   # e.g. (0, 1) to fix the density axis; None = auto-scaled to the data

In [8]:
t_check = times_kl[time_index_kl]
tag_check = f'{t_check}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'

with open(f'{n_latent_samples}_dataset_full_train_{tag_check}.pkl', 'rb') as f:
    df_full_check = dill.load(f)
with open(f'{n_latent_samples}_dataset_unique_train_{tag_check}.pkl', 'rb') as f:
    df_unique_check = dill.load(f)

design_point = df_unique_check.iloc[design_point_index_kl]
fck_check, rh_check, cov_check = design_point['fck'], design_point['rh'], design_point['cov']
g_real = df_full_check.loc[(df_full_check['fck'] == fck_check) & (df_full_check['rh'] == rh_check) & (df_full_check['cov'] == cov_check), 'g'].to_numpy()

print(f"Checking t = {t_check:.2f} years, fck = {fck_check:.3f}, rh = {rh_check:.3f}, cov = {cov_check:.3f} ({len(g_real)} raw g samples)")

kl_result_nn = validate_nn_kl_divergence_durability(
                                                       models=models,
                                                       scaler=scaler,
                                                       fck=fck_check,
                                                       rh=rh_check,
                                                       cov=cov_check,
                                                       t=t_check,
                                                       g_real=g_real,
                                                       lambda3=lambda3_kl,
                                                       lambda4=lambda4_kl,
                                                       n_grid=n_grid_kl,
                                                     )

print(f"KL divergence:  {kl_result_nn['kl_divergence']:.5f}")
print(f"KS statistic:   {kl_result_nn['ks_statistic']:.5f}")
print(f"Wasserstein:    {kl_result_nn['wasserstein']:.5f}")
print(f"R2 (PDFs):      {kl_result_nn['r2_pdf']:.5f}")
print(f"Rel. error P5:  {kl_result_nn['rel_err_p5']:+.2%}")
print(f"Rel. error P50: {kl_result_nn['rel_err_p50']:+.2%}")
print(f"Rel. error P95: {kl_result_nn['rel_err_p95']:+.2%}")

Checking t = 50.00 years, fck = 35.146, rh = 56.814, cov = 19.355 (2500 raw g samples)
KL divergence:  0.00892
KS statistic:   0.05360
Wasserstein:    0.05233
R2 (PDFs):      0.98142
Rel. error P5:  -4.35%
Rel. error P50: -3.33%
Rel. error P95: -0.56%


In [9]:
pkl_name_kl = f'{n_latent_samples}_dataset_unique_train_{tag_check}'

fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result_nn['x_grid'], kl_result_nn['pdf_real'], color='0.25', linewidth=2, label='Dataset')
ax.plot(kl_result_nn['x_grid'], kl_result_nn['pdf_nn'], color='crimson', linewidth=2, linestyle='--', label='NN-predicted GLD')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Probability density', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim_kl is not None:
    ax.set_xlim(xlim_kl)
if ylim_kl is not None:
    ax.set_ylim(ylim_kl)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, fontsize=tick_fontsize, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{pkl_name_kl}_nn_kl_divergence_fck{fck_check:g}_rh{rh_check:g}_cov{cov_check:g}_{t_check}_en.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

### 6.1 Mesmo gráfico, em português

In [10]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result_nn['x_grid'], kl_result_nn['pdf_real'], color='0.25', linewidth=2, label='Dados')
ax.plot(kl_result_nn['x_grid'], kl_result_nn['pdf_nn'], color='crimson', linewidth=2, linestyle='--', label='GLD previsto pela rede neural')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Densidade de probabilidade', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim_kl is not None:
    ax.set_xlim(xlim_kl)
if ylim_kl is not None:
    ax.set_ylim(ylim_kl)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, fontsize=tick_fontsize, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{pkl_name_kl}_nn_kl_divergence_fck{fck_check:g}_rh{rh_check:g}_cov{cov_check:g}_{t_check}_pt.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

### 6.2 Second time check — same design point, another $t$

In [11]:
time_index_kl_b = 4   # index into `times_kl` — the second time step to check (section 6 used `time_index_kl`)

lambda3_kl_b = lambda3_kl
lambda4_kl_b = lambda4_kl

In [12]:
t_check_b = times_kl[time_index_kl_b]
tag_check_b = f'{t_check_b}_install_{installation_year}_cement_{cement_type}_exposure_{exposure_conditions}_co2_{co2_scenario}'

with open(f'{n_latent_samples}_dataset_full_train_{tag_check_b}.pkl', 'rb') as f:
    df_full_check_b = dill.load(f)
with open(f'{n_latent_samples}_dataset_unique_train_{tag_check_b}.pkl', 'rb') as f:
    df_unique_check_b = dill.load(f)

design_point_b = df_unique_check_b.iloc[design_point_index_kl]
fck_check_b, rh_check_b, cov_check_b = design_point_b['fck'], design_point_b['rh'], design_point_b['cov']

if not (np.isclose(fck_check_b, fck_check) and np.isclose(rh_check_b, rh_check) and np.isclose(cov_check_b, cov_check)):
    print(f"WARNING: design point {design_point_index_kl} is not the same at both times")

g_real_b = df_full_check_b.loc[(df_full_check_b['fck'] == fck_check_b) & (df_full_check_b['rh'] == rh_check_b) & (df_full_check_b['cov'] == cov_check_b), 'g'].to_numpy()

print(f"Checking t = {t_check_b:.2f} years, fck = {fck_check_b:.3f}, rh = {rh_check_b:.3f}, cov = {cov_check_b:.3f} ({len(g_real_b)} raw g samples)")

kl_result_nn_b = validate_nn_kl_divergence_durability(
                                                         models=models,
                                                         scaler=scaler,
                                                         fck=fck_check_b,
                                                         rh=rh_check_b,
                                                         cov=cov_check_b,
                                                         t=t_check_b,
                                                         g_real=g_real_b,
                                                         lambda3=lambda3_kl_b,
                                                         lambda4=lambda4_kl_b,
                                                         n_grid=n_grid_kl,
                                                       )

print(f"KL divergence:  {kl_result_nn_b['kl_divergence']:.5f}")
print(f"KS statistic:   {kl_result_nn_b['ks_statistic']:.5f}")
print(f"Wasserstein:    {kl_result_nn_b['wasserstein']:.5f}")
print(f"R2 (PDFs):      {kl_result_nn_b['r2_pdf']:.5f}")
print(f"Rel. error P5:  {kl_result_nn_b['rel_err_p5']:+.2%}")
print(f"Rel. error P50: {kl_result_nn_b['rel_err_p50']:+.2%}")
print(f"Rel. error P95: {kl_result_nn_b['rel_err_p95']:+.2%}")

Checking t = 100.00 years, fck = 35.146, rh = 56.814, cov = 19.355 (2500 raw g samples)
KL divergence:  0.05428
KS statistic:   0.12000
Wasserstein:    0.25439
R2 (PDFs):      0.93541
Rel. error P5:  +3.62%
Rel. error P50: +5.07%
Rel. error P95: +10.55%


In [13]:
pkl_name_kl_b = f'{n_latent_samples}_dataset_unique_train_{tag_check_b}'

fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result_nn_b['x_grid'], kl_result_nn_b['pdf_real'], color='0.25', linewidth=2, label='Dataset')
ax.plot(kl_result_nn_b['x_grid'], kl_result_nn_b['pdf_nn'], color='crimson', linewidth=2, linestyle='--', label='NN-predicted GLD')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Probability density', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim_kl is not None:
    ax.set_xlim(xlim_kl)
if ylim_kl is not None:
    ax.set_ylim(ylim_kl)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, fontsize=tick_fontsize, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{pkl_name_kl_b}_nn_kl_divergence_fck{fck_check_b:g}_rh{rh_check_b:g}_cov{cov_check_b:g}_{t_check_b}_en.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

### 6.3 Mesmo gráfico, em português — segundo tempo

In [14]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result_nn_b['x_grid'], kl_result_nn_b['pdf_real'], color='0.25', linewidth=2, label='Dados')
ax.plot(kl_result_nn_b['x_grid'], kl_result_nn_b['pdf_nn'], color='crimson', linewidth=2, linestyle='--', label='GLD previsto pela rede neural')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Densidade de probabilidade', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim_kl is not None:
    ax.set_xlim(xlim_kl)
if ylim_kl is not None:
    ax.set_ylim(ylim_kl)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.18), ncol=2, fontsize=tick_fontsize, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{pkl_name_kl_b}_nn_kl_divergence_fck{fck_check_b:g}_rh{rh_check_b:g}_cov{cov_check_b:g}_{t_check_b}_pt.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

### 6.4 The two times side by side

In [15]:
stat_keys = {
                'KL':             'kl_divergence',
                'KS':             'ks_statistic',
                'Wasserstein':    'wasserstein',
                'R2 (PDF)':       'r2_pdf',
                'Rel. error P5':  'rel_err_p5',
                'Rel. error P50': 'rel_err_p50',
                'Rel. error P95': 'rel_err_p95',
            }

two_times_nn = pd.DataFrame([
                                {'Time (years)': t_c, 'fck': fck_c, 'rh': rh_c, 'cov': cov_c,
                                 **{label: res[key] for label, key in stat_keys.items()}}
                                for t_c, fck_c, rh_c, cov_c, res in [(t_check,   fck_check,   rh_check,   cov_check,   kl_result_nn),
                                                                     (t_check_b, fck_check_b, rh_check_b, cov_check_b, kl_result_nn_b)]
                            ])

two_times_nn

   Time (years)       fck  ...  Rel. error P50  Rel. error P95
0          50.0  35.14612  ...       -0.033278       -0.005641
1         100.0  35.14612  ...        0.050684        0.105477

[2 rows x 11 columns]

In [16]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result_nn['x_grid'],   kl_result_nn['pdf_real'],   color='0.25',    linewidth=2,                 label=f'Dataset, $t = {t_check:g}$ yr')
ax.plot(kl_result_nn['x_grid'],   kl_result_nn['pdf_nn'],     color='crimson', linewidth=2, linestyle='--', label=f'NN, $t = {t_check:g}$ yr')
ax.plot(kl_result_nn_b['x_grid'], kl_result_nn_b['pdf_real'], color='#2a78d6', linewidth=2,                 label=f'Dataset, $t = {t_check_b:g}$ yr')
ax.plot(kl_result_nn_b['x_grid'], kl_result_nn_b['pdf_nn'],   color='#eb6834', linewidth=2, linestyle='--', label=f'NN, $t = {t_check_b:g}$ yr')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Probability density', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim_kl is not None:
    ax.set_xlim(xlim_kl)
if ylim_kl is not None:
    ax.set_ylim(ylim_kl)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.28), ncol=2, fontsize=tick_fontsize - 2, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{n_latent_samples}_nn_kl_divergence_fck{fck_check:g}_rh{rh_check:g}_cov{cov_check:g}_t{t_check:g}_vs_t{t_check_b:g}_en.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()

In [17]:
fig, ax = plt.subplots(figsize=fig_size)
ax.plot(kl_result_nn['x_grid'],   kl_result_nn['pdf_real'],   color='0.25',    linewidth=2,                 label=f'Dados, $t = {t_check:g}$ anos')
ax.plot(kl_result_nn['x_grid'],   kl_result_nn['pdf_nn'],     color='crimson', linewidth=2, linestyle='--', label=f'Rede neural, $t = {t_check:g}$ anos')
ax.plot(kl_result_nn_b['x_grid'], kl_result_nn_b['pdf_real'], color='#2a78d6', linewidth=2,                 label=f'Dados, $t = {t_check_b:g}$ anos')
ax.plot(kl_result_nn_b['x_grid'], kl_result_nn_b['pdf_nn'],   color='#eb6834', linewidth=2, linestyle='--', label=f'Rede neural, $t = {t_check_b:g}$ anos')
ax.set_xlabel('$g$', fontsize=label_fontsize)
ax.set_ylabel('Densidade de probabilidade', fontsize=label_fontsize)
ax.tick_params(axis='both', labelsize=tick_fontsize)
if xlim_kl is not None:
    ax.set_xlim(xlim_kl)
if ylim_kl is not None:
    ax.set_ylim(ylim_kl)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, 1.28), ncol=2, fontsize=tick_fontsize - 2, frameon=False)
ax.grid(True, alpha=0.3)
fig.tight_layout()

fig.savefig(f'{n_latent_samples}_nn_kl_divergence_fck{fck_check:g}_rh{rh_check:g}_cov{cov_check:g}_t{t_check:g}_vs_t{t_check_b:g}_pt.{fig_format}', bbox_inches='tight', dpi=fig_dpi)
plt.show()